# F7-kernels-convex-optimization — Session 2: Kernel Proofs and Counterexamples

**85 minutes.** A valid kernel must produce a PSD Gram matrix for every finite input list. Today we build universal proofs from feature maps and closure rules, then learn how one negative eigenpair becomes a constructive disproof.


## 1. Start every argument with the quantifier

To prove `k` valid, fix an arbitrary finite list `x_1,...,x_n` and arbitrary coefficients `a_1,...,a_n`, then prove

$$\sum_{i,j} a_i a_j k(x_i,x_j) >= 0.$$

Nothing about the proof may depend on a special dataset size or lucky sample. To disprove validity, reverse the burden: one explicit finite list and one coefficient vector with negative quadratic form suffice.

This asymmetry explains the roles of computation. A numerical PSD result is local evidence. A reliable negative eigenpair, checked through `a^TKa<0`, is a universal counterexample because validity quantified over that list too.

### Checkpoint 1

1. Write the first sentence of a universal kernel proof for an arbitrary list and coefficient vector.
2. What two concrete objects must a constructive invalidity certificate name?


## 2. Direct feature-map proofs

If `k(x,z)=φ(x)^Tφ(z)`, then for arbitrary coefficients,

$$\sum_{i,j}a_i a_j k(x_i,x_j)=||\sum_i a_i φ(x_i)||_2^2>=0.$$

That single squared norm handles every finite list. Useful base kernels include:

- constant: `k(x,z)=c` for `c>=0`, with feature `sqrt(c)`;
- linear: `k(x,z)=x^Tz`, with `φ(x)=x`;
- rank-one weighted: `k(x,z)=f(x)f(z)`, with scalar feature `f(x)`.

A feature map proves PSD, not strict PD. Different inputs may share a feature vector, and repeated inputs certainly do.

### Checkpoint 2

1. Give a one-coordinate feature map for `k(x,z)=9x^2z^2` on scalar inputs.
2. Why can a perfectly valid finite-dimensional feature map yield a singular Gram matrix when `n` exceeds the feature dimension?


## 3. Closure rules you can prove, not memorize

Suppose `k_1` and `k_2` are valid.

**Nonnegative scaling.** For `c>=0`, `c k_1` is valid because its quadratic form is `c` times a nonnegative number. Negative scaling has no such guarantee.

**Sum.** `k_1+k_2` is valid because its quadratic form is the sum of two nonnegative forms. With feature maps, concatenate the feature vectors.

**Product.** On an arbitrary finite list, let the Gram matrices be `K_1` and `K_2`. F6's PSD factorization writes `K_2=B B^T`. If `b_r` is column `r` of `B` and `D_r=diag(b_r)`, then

$$K_1\circ K_2=\sum_r D_r K_1 D_r.$$

Every term is PSD because $a^TD_rK_1D_ra=(D_ra)^TK_1(D_ra)\ge0$. Thus the entrywise product is PSD on every finite list. When explicit feature maps `φ` and `ψ` are available, the same rule appears as the tensor feature `χ(x)_(r,s)=φ_r(x)ψ_s(x)`.

**Input transformation.** If `k` is valid and `T` is any fixed map into its input domain, then `k(T(x),T(z))` is valid: every transformed finite list is still a finite list covered by `k`.

Subtraction is absent. The difference of two PSD matrices need not be PSD.

### Checkpoint 3

1. Prove `3k_1+5k_2` valid in one line using quadratic forms.
2. Which closure rule proves `(1+xz)^2` valid once `1+xz` is known valid?


## 4. Polynomial kernels, coefficient by coefficient

For vectors `x,z`, the inhomogeneous polynomial kernel

$$k(x,z)=(c+x^Tz)^m$$

is valid when `c>=0` and `m` is a nonnegative integer. The constant kernel `c` and linear kernel `x^Tz` are valid; their sum is valid; repeated products preserve validity.

**Worked exam-style example 1.** For `x=(x_1,x_2)` and degree two,

$$(1+x^Tz)^2=1+2x_1z_1+2x_2z_2+x_1^2z_1^2+2x_1x_2z_1z_2+x_2^2z_2^2.$$

An explicit feature map is

$$φ(x)=(1, sqrt(2)x_1, sqrt(2)x_2, x_1^2, sqrt(2)x_1x_2, x_2^2).$$

Every coefficient is chosen so the dot product reproduces the expansion. This proves validity for arbitrary inputs; a finite numerical table merely checks the algebra.

### Checkpoint 4

1. How many coordinates does the displayed degree-two map have?
2. Why does the proof require integer `m` rather than treating an arbitrary real power as repeated multiplication?


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

X = np.array([[1.0, 2.0], [-1.0, 1.0], [0.5, -2.0]])
Phi = np.column_stack([
    np.ones(len(X)),
    np.sqrt(2.0) * X[:, 0],
    np.sqrt(2.0) * X[:, 1],
    X[:, 0] ** 2,
    np.sqrt(2.0) * X[:, 0] * X[:, 1],
    X[:, 1] ** 2,
])
K_feature = Phi @ Phi.T
K_formula = (1.0 + X @ X.T) ** 2
assert np.allclose(K_feature, K_formula, atol=ATOL, rtol=RTOL)
assert np.linalg.eigvalsh(K_formula).min() >= -ATOL


## 5. Finite tests: useful evidence, never a universal proof

A sound numerical audit of one proposed Gram matrix checks:

1. every entry is finite;
2. symmetry holds within explicit tolerance;
3. `np.linalg.eigvalsh` is used for the symmetric spectrum; and
4. any claimed negative eigenvalue is large compared with the declared tolerance and has a matching quadratic-form witness.

A thousand passing random samples still leave infinitely many untested lists. Numerical search is excellent for discovering a counterexample and for checking a symbolic derivation. It cannot replace the universal feature-map or closure argument.

### Checkpoint 5

1. Why is `eigvalsh` preferable to a general eigensolver after symmetry is established?
2. State one legitimate conclusion from 100 passing random Gram tests and one illegitimate conclusion.


## 6. Constructive negative-eigenvalue counterexamples

Consider the symmetric candidate `k_bad(x,z)=1-xz` on scalar inputs. On `x_1=-1` and `x_2=1`,

$$K=[[0,2],[2,0]].$$

Its eigenvalues are `-2` and `2`. More constructively, with `a=(1,-1)`,

$$a^TKa=-4<0.$$

That finite list and coefficient vector disprove validity. The failure also diagnoses a tempting algebra error: `1` and `xz` are valid, but closure allows nonnegative sums, not subtraction.

**Worked exam-style example 2.** A candidate passes Gram tests on `[0,1]` but fails on `[-1,1]`. The correct response is not an empirical confidence score. Report the failing inputs, the matrix, and either its negative eigenpair or a coefficient vector with negative energy. That is a complete mathematical refutation.

### Checkpoint 6

1. Verify `a^TKa=-4` without an eigensolver.
2. Why would clipping the negative eigenvalue to zero destroy the counterexample rather than repair the kernel?


In [ ]:
x_bad = np.array([-1.0, 1.0])
K_bad = 1.0 - x_bad[:, None] * x_bad[None, :]
evals, evecs = np.linalg.eigh(K_bad)
witness = evecs[:, 0]
energy = witness @ K_bad @ witness
assert np.allclose(K_bad, np.array([[0.0, 2.0], [2.0, 0.0]]), atol=ATOL, rtol=RTOL)
assert np.isclose(evals[0], -2.0, atol=ATOL, rtol=RTOL)
assert np.isclose(energy, evals[0], atol=ATOL, rtol=RTOL)
assert energy < -ATOL


## 7. Common pitfalls, exam connections, and forward links

**Pitfall — closure under subtraction.** Broken move: valid minus valid is valid. Fix: only use a listed rule with its sign assumptions, or build a new feature map.

**Pitfall — the formula looks symmetric, so it is a kernel.** Symmetry is necessary but not sufficient; `1-xz` is symmetric and invalid.

**Pitfall — repeated inputs make a zero eigenvalue, so reject.** Zero is allowed for PSD. A counterexample needs a genuinely negative quadratic form.

**Exam connection.** Expect a proof/counterexample choice: construct a feature map or closure chain for a valid candidate; provide a small explicit Gram witness for an invalid one. A bare line of NumPy output is not the reasoning deliverable.

**Going deeper.** Future SVM work uses kernels as interchangeable similarity engines. The validity proof belongs here; the later algorithm may assume it.

### Checkpoint 7

1. Is `k(x,z)=4+(x^Tz)^3` valid on real vectors? Name the closure chain.
2. Is a symmetric candidate with one tested PSD Gram matrix proven valid? Answer with the exact missing quantifier.


## Checkpoint answers

<details><summary><b>Checkpoint 1</b></summary>

1. Fix arbitrary finite inputs and arbitrary real coefficients. 2. A finite input list and a coefficient vector (or negative eigenvector) with negative quadratic form.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `φ(x)=3x^2`. 2. The Gram rank is at most the feature dimension, so more rows force dependence and zero eigenvalues.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Its quadratic form is three times a nonnegative form plus five times another. 2. Product closure.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Six. 2. Repeated product closure describes nonnegative integer powers; arbitrary real powers need a separate argument and may not even be real-valued everywhere.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. It exploits symmetry and returns a real ordered spectrum. 2. Legitimate: those 100 matrices were PSD within tolerance. Illegitimate: every finite Gram matrix is PSD.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. `K(1,-1)^T=(-2,2)^T`, whose dot with `(1,-1)` is `-4`. 2. Clipping changes the matrix/candidate; it does not prove the original quadratic form nonnegative.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Yes: linear kernel, product closure three times, nonnegative constant, then sum. 2. No; validity requires PSD for every finite input list.

</details>
